# 05 · Launch PyTorch Lightning — Multi Node

Multi-node 토폴로지(M×N). 학습 함수는 [`lightning_trainer.py`](./lightning_trainer.py)의 `fit(...)`이며 1×1·1×N과 동일합니다. 차이는 `num_nodes=M`과 worker 클러스터, 그리고 TorchDistributor `local_mode=False`.

> Multi-node에서는 Lightning의 `ddp_notebook`이 동작하지 않으므로 TorchDistributor가 worker process를 띄우고, 각 process 안에서 `Trainer(strategy='ddp')`가 이미 세팅된 `RANK`/`WORLD_SIZE`/`MASTER_ADDR`를 그대로 사용합니다.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML. Driver `g5.12xlarge` + Workers `g5.12xlarge` × M. **Autoscaling OFF 필수**, Single node 토글 OFF. Serverless GPU 사용 불가.

In [ ]:
%run ./00-setup

## 학습 함수 import + SCRIPT_DIR

In [ ]:
import os
import sys

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")

from lightning_trainer import fit

## M×N GPU

`NUM_NODES`, `NUM_GPUS_PER_NODE`를 클러스터 구성에 맞춥니다. 1×1·1×N과 같은 `CONFIG`·같은 `DATA_DIR`.

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2          # M
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lit-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    fit,
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lit-MxN"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    devices=NUM_GPUS_PER_NODE,
    num_nodes=NUM_NODES,
    topology="MxN",
    script_dir=SCRIPT_DIR,
)